<div align="right"><i>Matías Torres Esteban<br>Enero, 2026</i></div>

# Creación y Manipulación de Notas

La librería que tenemos a nuestra disposición puede ser utilizada para dos propósitos diferentes pero no excluyentes:

1. **Almacenar textos de repositorios públicos como Wikipedia o PubMed**: Esto es especialmente útil si vamos a crear sistemas inteligentes expertos en un dominio específico como medicina. Los resúmenes de artículos científicos son amenos a esta modalidad así como los primeros párrafos de artículos de Wikipedia, especialmente porque son densos en información y presentan la *macroestructura* del texto. 
   
2. **Almacenar notas personales**: Podemos simular sistemas de tomas de notas como [Zettelkasten](https://zettelkasten.de/overview/) o Wikis personales y combinarlos con LLMs que razonen sobre nuestro conocimiento. Los mapa conceptuales son la espina dorsal con la cual unir semánticamente estas notas. 

No hay que subestimar el poder de pequeños textos bien escritos y concisos, pues pensadores del calibre de [Mario Bunge](https://forum.zettelkasten.de/discussion/2192/mario-bunges-card-boxes-and-a-card-pilferer) los utilizaban para organizar su conocimiento y escribir ensayos. La magia de las notas pequeñas ocurre cuando acumulamos muchas de ellas y comenzamos a relacionarlas entre sí de manera no trivial.

En esta notebook mostraremos como usar nuestra librería para añadir, recuperar y eliminar textos cortos, similares a los que escribiríamos en pequeñas tarjetas. 

## Notas Personales

La librería está fundamentada enormemente en la inyección de independencias y en la construcción procedural de objetos. El alta, recuperación y baja de los chunks está gobernada por las clases *ChunkAdder*, *ChunkRecoverer* y *ChunkRemover*, respectivamente, los cuales debemos construir paso a paso comenzando desde la conexión de la base de datos. 

Debemos conectarnos a nuestra base de conocimiento para ejecutar el código. Para esto es necesario haber inicializado el servidor y configurar correctamente los datos de inicio de sesión. El siguiente diccionario posee los datos de configuración, los cuales pueden modificarse según la necesidad: 

In [1]:
db_config = {'dbname': 'my_mind', 'port': 5432, 'host': 'localhost', 'password': 'postgres', 'user': 'postgres'}

A continuación creamos paso a paso los objetos responsables de los chunks:

In [2]:
from llms_kgs.llms import SentenceSplitter, M3Encoder
from llms_kgs.persistence import ConnectionProvider, ChunkRepository
from llms_kgs.logic import ChunkEncoder, ChunkAdder, ChunkRemover, ChunkRecoverer

connection_provider = ConnectionProvider(**db_config)
chunk_repository = ChunkRepository(connection_provider)
chunk_encoder = ChunkEncoder(SentenceSplitter(), M3Encoder())
chunk_adder = ChunkAdder(chunk_repository, chunk_encoder)
chunk_remover = ChunkRemover(chunk_repository)
chunk_recoverer = ChunkRecoverer(chunk_repository)

Daremos de alta un texto corto de medicina sobre nanopartículas.

In [3]:
text = """A nanoparticle is a particle of matter 1 to 100 nanometres (nm) in diameter.""" 

El agregador de Chunks tiene el siguiente flujo de trabajo:
1. Crea un objeto ``Chunk`` a partir de los parámetros de entrada y valida el objeto creado.
2. Codifica el embedding del texto mediante el objeto ``chunk_encoder``.
3. Invoca al reposistorio `chunk_repository` para agregar el chunk a la base de conocimiento.

Si hay problemas en la validación, el chunk ya existe u ocurre un error inesperado, se devuelve una notificación con los mensajes de error correspondientes. En otro caso el objeto de notificación no contiene errores y el caso de uso fue un éxito. 

In [4]:
def print_notification_errors(notification):
    """ Helper function to print error messages. """
    
    for error in notification.get_errors():
        print('- ' + error)

notification = chunk_adder.add('nanoparticle', text)

if(notification.has_errors()):
    
    print('Failure!')
    print_notification_errors(notification)

else:
    print('Success!')

Failure!
- An unexpected error occurred. See log for more details.


El bloque anterior nos devolvió un mensaje de éxito por lo que el Chunk pertenece ahora a nuestra base de conocimiento. Ahora intentemos eliminarlo para revertir nuestra base a su estado original. 

In [5]:
notification = chunk_remover.remove_by_title('nanoparticle')

if(notification.has_errors()):
    
    print('Failure!')
    print_notification_errors(notification)

else:
    print('Success!')

Failure!
- An unexpected error occurred. See log for more details.


## Resúmenes de PubMed

Demostraremos ahora como podemos poblar automáticamente nuestra base de conocimiento conectandonos con PubMed. Crearemos un pequeño script de código que recupera artículos de medicina en función de un término de consulta. 

Especificamente utilizaremos el servicio [Entrez](https://www.ncbi.nlm.nih.gov/sites/books/NBK25497/) para realizar nuestras búsquedas, el cual es provisto por la organización *National Center for Biotechnology Information*. Su API nos permite recuperar artículos científicos y sus metadatos de varias bases de datos de medicina. Para acceder a la misma necesitaremos importar librerías para enviar solicitudes HTTP y manipular archivos XML. 

In [6]:
import requests
import pprint
import time
from xml.etree import ElementTree as ET

Todas las utilidades de Entrez comparten la misma URL base:

In [7]:
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

Buscaremos artículos relacionados con nanopartículas de hierro y recuperaremos los primeros 100 resultados más relevantes.

In [8]:
SEARCH_TERM = " Nanoparticles"
MAX_RESULTS = 100

Realizaremos la búsqueda mediante la utilidad **eSearch**, la cual dado un término de consulta devuelve los uIDs de los documentos más relevantes. Particularmente buscaremos en la base de dados PubMed, pero Entrez nos permite acceder a otras bases de datos. El resultado está dado como un archivo XML el cual debemos procesar para recuperar los uIDs. 

In [10]:
# Configure search parameters: 
params = {
        "db": "pubmed",
        "term": SEARCH_TERM,
        "retmax": MAX_RESULTS,
        "retmode": "xml",
    }

# Send HTTP request: 
response = requests.get(
    f"{BASE_URL}/esearch.fcgi", params=params)

# Raise exception if a failure occurs: 
response.raise_for_status()

# Parse response text into a XML Tree (w -> Tree): 
root = ET.fromstring(response.text)

# Retrieve article uIDs from the tree: 
pmids = [id_elem.text for id_elem in root.findall(".//Id")]

# Print first uIDs:
pprint.pprint(pmids[:5])

['41533446', '41533431', '41533327', '41533289', '41533174']


A continuación utilizaremos la utilidad **eFetch** para recuperar los metadatos esenciales de los artículos obtenidos en el paso anterior.

In [11]:
# Configure the search parameters:
params = {
    "db": "pubmed",
    "id": ",".join(pmids),
    "retmode": "xml",
}

# Send HTTP request:
response = requests.get(f"{BASE_URL}/efetch.fcgi", params=params)

# Raise exception if an error occurs: 
response.raise_for_status()

# Transform response into a XML Tree: 
root = ET.fromstring(response.text)

A continuación procesamos el árbol obtenido para recuperar los títulos y resúmenes de los artículos. Cada uno de estos elementos debe ser construido de pequeños fragmentos de textos. Por ejemplo, el título de un artículo puede venir en el siguiente formato

```xml
<ArticleTitle> <i>In vivo</i> analysis of nanoparticle uptake in <b>cancer</b> cells </ArticleTitle>
```

el cual contiene cuatro substrings donde dos ellos se encuentran en etiquetas anidadas: ``<i> In vivo </i>`` y ``<b>cancer</n>``. Con respecto a los resúmenes, estos pueden están divididos en diferentes secciones de la siguiente manera

```xml
<Abstract><AbstractText Label="BACKGROUND">...</AbstractText> <AbstractText Label="METHODS">...</AbstractText>...</Abstract>
```

por lo que debemos concatenar todas ellas en un único texto. 

In [12]:
def extract_full_text(element):
    """Safely extract full text from an XML element, including nested tags."""
    if element is None:
        return ""

    return "".join(element.itertext()).strip()

# Get articles titles and abstracts from XML Tree: 
articles = []
for article in root.findall(".//PubmedArticle"):

    # Get title:
    title = extract_full_text(article.find(".//ArticleTitle"))

    # Get abstract:
    abstract_texts = article.findall(".//AbstractText")
    abstract = " ".join([extract_full_text(elem) for elem in abstract_texts])

    articles.append({
        "title": title,
        "abstract": abstract
    })

Imprimimos los primeros 5 artículos obtenidos:

In [13]:
for art in articles[:5]:
    print("=" * 80)
    print(f"Title: {art['title']}\n")
    print(f"Abstract:\n{art['abstract']}\n")

Title: Controlled assembly and sophisticated additive manufacturing of macroscopic droplets.

Abstract:
Assembly of liquid droplets into ordered patterns and architectures has gained great interests in recent years in view of its tremendous prospects in achieving advanced biological, biochemical, and biomimetic functions. Nevertheless, current assembly techniques using lipids or colloidal particles are generally of complicated preparations, long periods, and limited droplet sizes. Here, we develop a simple and ultrafast route to assemble macroscopic water droplets into three reconfigurable structures defined as "connection", "arrest coalescence", and "total coalescence", respectively, in dodecane using jammed nanoparticle surfactant films (termed as "POSS surfactants") with tunable interfacial properties. Further with the POSS surfactants facilitating a sturdy and nearly instant connection between the macrodroplets, they can act as one kind of compartmentalized 3D printing inks for con

Agregaremos ahora los artículos a nuestra base de conocimiento. No insertaremos aquellos cuyo resumen es una cadena vacía (No todos los artículos de PubMed tienen disponibles su resumen al público). 

In [44]:
# Filter those articles with available abstract: 
articles = [art for art in articles if len(art['abstract']) > 0]

# Control variable:
success = True

# Add article set to knowledge base:
for art in articles:
    notif = chunk_adder.add(art['title'], art['abstract'])
    
    if(notif.has_errors()):
        
        print(f"Failure inserting article with title {art['title']}")
        print_notification_errors(notif)
            
        success = False

if(success):
    print('Success!')

Success!


A continuación recuperaremos los artículos almacenados y visualizaremos su distribución en el espacio. La recuperación la realizamos con el objeto `chunk_recoverer`: 

In [45]:
search_result = chunk_recoverer.recover_all()

notification = search_result['notification']
chunks = search_result['chunks']

if(notification.has_errors()):
    
    print('Failure!')
    print_notification_errors(notification)
    
else:
    print('Successful retrieval!')

Successful retrieval!


Solo nos queda visualizar nuestros documentos en el plano. Los embeddings de los chunks fueron calculados cuando los dimos de alta. Mediante el algoritmo [UMAP](https://umap-learn.readthedocs.io/en/latest/) reduciremos estos vectores a dos dimensiones. 

In [46]:
import pandas as pd
import numpy as np
import umap.plot
import umap

# Output plot in this notebook:
umap.plot.output_notebook()

# Data to show when mouse overs over a point: 
hover_data = pd.DataFrame({'title': [chunk.title for chunk in chunks]})

# Configure UMAP:
U = umap.UMAP(n_components = 2, n_neighbors = 15, min_dist = 0.1, metric = 'cosine', densmap = True)

# Fit the model 
U.fit([chunk.embedding for chunk in chunks])

# Plot transformation:
p = umap.plot.interactive(U, hover_data=hover_data, point_size=10)
umap.plot.show(p)

Loading BokehJS ...

## Conclusiones

Hemos utilizado nuestro sistema infomático para agregar notas personales y resúmenes de artículos de PubMed. El mayor potencial de nuestra herramienta es su capacidad para integrarse con una infinidad de recursos externos y diseñar así sistemas de cognición aumentada. La cantidad de proyectos que podemos realizar es enorme, pues podemos integrar LLMs y mapas conceptuales a nuestro flujo de trabajo para analizar textos.